#### Imports

In [1]:
import pandas as pd
import os
import mediacloud.api
from waybacknews.searchapi import SearchApiClient
import datetime as dt
import time
from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()

True

#### Load Shortlisted Sources
- Extract list of source IDS

In [2]:
df_sources = pd.read_csv('../Source Selection/ShortlistedSources.csv')

source_IDs = list(df_sources['MediaCloud Source IDs'])
domains = list(df_sources['Domains'])

#### Setup MediaCloud Environment

In [4]:
my_api_key = os.environ.get('MEDIACLOUD_API_KEY')
search_api = mediacloud.api.SearchApi(my_api_key)

start = dt.date(2005,1,1)
end = dt.date(2025,6,30)

In [19]:
def get_stories(MC_API_KEY:str, source_ids:list, query:str, start_date, end_date, count:int):
    search_api = mediacloud.api.SearchApi(MC_API_KEY)

    all_stories = []
    more_stories = True
    pagination_token = None


    with tqdm(total=count, desc="Fetching stories", unit="story") as pbar:
        while more_stories:

            time.sleep(30) # API rate limit - 2 requests every minute

            page, pagination_token = search_api.story_list(query, 
                                                        start_date, 
                                                        end_date,
                                                        source_ids=source_IDs,
                                                        pagination_token=pagination_token)
            all_stories += page
            pbar.update(len(page))

            more_stories = pagination_token is not None
    
    return pd.DataFrame(all_stories)

#### Extract Stories
- Organised by event (and/or series of events) first
- Stored in a dataframe for each event

### General Elections

In [20]:
my_query = '''

(india OR indian OR indians)

AND 

("national elections" OR "national election" OR "general elections" OR "general election" 
OR "lok sabha elections" OR "lok sabha election")

AND

(poll OR polls OR polling OR bypoll*)

AND 

(vote OR vote? OR voting OR ballot*)

AND

(

    (party OR campaign OR centre OR opposition)

    OR

    ("election commission*" OR EC OR ECI OR EVM OR VVPAT)

)

AND NOT (cricket OR bollywood OR "stock market")

'''

In [8]:
start_dt = dt.datetime.combine(start, dt.time(0,0,0))
end_dt = dt.datetime.combine(dt.date(2018,12,31), dt.time(0,0,0))

results = SearchApiClient("mediacloud").count(my_query + "AND domain:({})".format(" OR ".join(domains)), start_dt, end_dt)

print(results)

28


In [ ]:
stories = SearchApiClient("mediacloud").all_articles(my_query + "AND domain:({})".format(" OR ".join(domains)), start_dt, end_dt)

init_sample = next(stories)

init_sample

In [21]:
results = search_api.story_count(my_query, start, end, source_ids=source_IDs)

print(results)

{'relevant': 31685, 'total': 10912206}


In [22]:
df_elections = get_stories(my_api_key, source_IDs, my_query, start, end, results['relevant'])

df_elections

Fetching stories: 31686story [17:18, 30.51story/s]                        


,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,8d22578f9f79e150731383527a7ef00a3eab056ad041b7...,2025-07-08 14:58:32.927873+00:00,en,dnaindia.com,dnaindia.com,2019-02-13,Lok Sabha polls 2019: Opposition to have commo...,https://www.dnaindia.com/india/report-lok-sabh...
1,d9841f5a851d29a67616e579ed52bbda9f85a2083ab2ac...,2025-07-08 14:49:59.495216+00:00,en,indianexpress.com,indianexpress.com,2019-02-14,BJP may allow Sena play big brother in state p...,https://indianexpress.com/elections/bjp-may-al...
2,175e9165d1b06fb5b9d43eac96ba27ba2cd7e6684190d7...,2025-07-08 14:39:40.303629+00:00,en,indiatoday.in,indiatoday.in,2019-02-08,Tamil Nadu,https://www.indiatoday.in/magazine/states/stor...
3,1f73948591b01fd10d07191a908fcf9d4a695935677218...,2025-07-08 14:29:56.171953+00:00,en,hindustantimes.com,hindustantimes.com,2019-02-14,"All about Mahan Dal, the party Congress firmed...",https://www.hindustantimes.com/india-news/all-...
4,37fbe7d1f2d9624ea093bbad5382760283b48753e6b249...,2025-07-08 14:21:32.815019+00:00,en,ndtv.com,ndtv.com,2019-02-14,"Mayawati Blasts Congress Again, Delivers Messa...",https://www.ndtv.com/india-news/lok-sabha-elec...
...,...,...,...,...,...,...,...,...
31681,4b5a7fdb681e9e96ff8c3285e3beb68d91a525b86d9c81...,2023-11-12 00:13:15+00:00,en,indiatimes.com,indiatimes.com,2023-11-10,"Ensure Elections Are Inducement-free, Safe, Ec...",https://timesofindia.indiatimes.com/city/chenn...
31682,5504401069c3cc183c56f183362693471ee490a817b835...,2023-11-12 00:12:22+00:00,en,indiatimes.com,indiatimes.com,2023-11-10,"Aimim Corporator Stops Rally, Booked",https://timesofindia.indiatimes.com/city/hyder...
31683,719429fdcbd95cc80a29daffa8079b566693fb465c55dc...,2023-11-12 00:12:22+00:00,en,indiatimes.com,indiatimes.com,2023-11-10,Inter-school Band Contest Kicks Off In City,https://timesofindia.indiatimes.com/city/ranch...
31684,4ad970e341448a2b0972b4afcef056551c88755c3e2d11...,2023-11-12 00:12:20+00:00,en,indiatimes.com,indiatimes.com,2023-11-10,Scindia Toppled Govt To Fulfil Friend Raga’s P...,https://timesofindia.indiatimes.com/city/indor...


In [8]:
df_elections.to_csv('elections.csv')

### Union Budget

In [9]:
my_query = '''
(india OR indian OR indians) 

AND 

("budget 20??" OR "union budget" OR "central budget" OR "annual financial statement")

AND 

(tax* OR GDP OR financ* OR econom* OR FY OR fiscal) 

AND NOT (cricket OR bollywood)

'''

In [10]:
results = search_api.story_count(my_query, start, end, source_ids=source_IDs)

print(results)

{'relevant': 19003, 'total': 10420074}


In [11]:
df_budget = get_stories(my_api_key, source_IDs, my_query, start, end, results['relevant'])

df_budget

Fetching stories: 19037story [10:40, 29.72story/s]                        


,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,dcb0aeace54a4eff23c9fa6ea824da195afe2dbbff6148...,2025-07-02 15:45:32.191912+00:00,en,hindustantimes.com,hindustantimes.com,2019-06-20,An opposition-mukt India is proof of a looming...,https://www.hindustantimes.com/columns/an-oppo...
1,b88d43ee85abb1ae25e00529c7528157f69f149d05b4d9...,2025-07-02 15:27:50.639512+00:00,en,ndtv.com,ndtv.com,2019-06-21,"Clarity On Taxes, Compliance To Boost Investor...",https://www.ndtv.com/business/budget-2019-clar...
2,761e9e51c74e2de14616846b35710b74a80c1904cd35a9...,2025-07-02 15:23:01.170709+00:00,en,indiatimes.com,indiatimes.com,2019-06-21,FM Nirmala Sitharaman holds pre-budget meeting...,https://economictimes.indiatimes.com/news/econ...
3,2a4e323276083edfedd6385d4e4b5aafc9e28bb0c7bada...,2025-07-02 15:22:24.552426+00:00,en,india.com,india.com,2019-06-21,Sitharaman Meets State Finance Ministers For P...,https://www.india.com/business/finance-ministe...
4,a901e251ff16717c7d9f2b6449c6343abbb7c310daa1eb...,2025-07-02 15:16:46.819178+00:00,en,hindustantimes.com,hindustantimes.com,2019-06-21,"Not Rs 325 cr, give us Rs 6,000 cr, Kejriwal’s...",https://www.hindustantimes.com/india-news/not-...
...,...,...,...,...,...,...,...,...
19032,646292395e078f075961bb81006a3b38cda417118efc9e...,2023-11-15 00:23:31+00:00,en,india.com,india.com,2023-11-13,PM Modi To Launch PM PVTG (Particularly Vulner...,https://www.india.com/news/india/pm-modi-to-la...
19033,16c7f0238069bef4718107bc6cd6e5d3b47366b027f104...,2023-11-13 01:44:12+00:00,en,ndtv.com,ndtv.com,2023-11-11,National Education Day: How Is India Leading I...,https://www.ndtv.com/education/national-educat...
19034,2d7f277cf182f2e2c9c41a7d3516d6ec6f22205cf225f5...,2023-11-13 00:09:45+00:00,en,indiatimes.com,indiatimes.com,2023-02-07,GST on digital services: Budget 2023-24 broade...,https://economictimes.indiatimes.com/small-biz...
19035,7753d0a284ba8f43b17bfa822a5a09068267fdf22f7ca9...,2023-11-12 00:15:06+00:00,en,thehindu.com,thehindu.com,2023-03-03,Programme to upskill construction workers,https://www.thehindu.com/life-and-style/homes-...


In [12]:
df_budget.to_csv('budget.csv')

### Supreme Court Judgements

In [13]:
my_query = '''
(india OR indian OR indians) 

AND

("supreme court judgement" OR "supreme court verdict" OR "supreme court ruling" OR "supreme court decision" OR "supreme court hear*"
OR "supreme court order" OR "SC judgement" OR "SC verdict" OR "SC ruling" OR "SC decision" OR "SC order" OR "SC hear*"
OR "apex court judgement" OR "apex court verdict" OR "apex court ruling" OR "apex court decision" OR "apex court order" OR "apex court hear*"
OR "top court judgement" OR "top court verdict" OR "top court ruling" OR "top court decision" OR "top court order" OR "top court hear*")

AND

(article OR section OR bill OR case OR petition OR plea OR appeal OR appeals)

AND NOT (cricket OR bollywood)
'''

In [14]:
results = search_api.story_count(my_query, start, end, source_ids=source_IDs)

print(results)

{'relevant': 11959, 'total': 10420074}


In [15]:
df_judgements = get_stories(my_api_key, source_IDs, my_query, start, end, results['relevant'])

df_judgements

Fetching stories: 11965story [06:32, 30.50story/s]                        


,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,737c10183dda274518a7cefef441d70d9ac81a46f3e6c6...,2025-07-02 16:02:39.950938+00:00,en,indianexpress.com,indianexpress.com,2019-06-21,Maharashtra: House clears Bill on Maratha quot...,https://indianexpress.com/article/india/mahara...
1,7082899e0e9e9b8e7cc111705c6dc4a9d171b88f876db8...,2025-07-02 15:21:36.170357+00:00,en,indiatimes.com,indiatimes.com,2019-06-21,View: Here's a treatment for mob violence agai...,https://economictimes.indiatimes.com/news/poli...
2,7af20dfe85b49fedc84d4e439b3dafe9a3274c77d0eb18...,2025-07-02 14:39:26.799065+00:00,en,thehindu.com,thehindu.com,2019-06-21,Sabarimala Bill seen as political posturing,https://www.thehindu.com/news/national/kerala/...
3,e0ce033fb91ee4ea19511f3d29d05579254e7f3bfbc642...,2025-07-02 14:39:26.488186+00:00,en,thehindu.com,thehindu.com,2019-06-21,IRDAI okays standalone own-damage policy,https://www.thehindu.com/business/irdai-okays-...
4,8add55f94311a5c30ba8ef451a34f235e9e0877f2f9bdb...,2025-07-02 14:05:57.522834+00:00,en,hindustantimes.com,hindustantimes.com,2019-06-22,SC puts Chhattisgarh govt in fix over CBI prob...,https://www.hindustantimes.com/india-news/sc-p...
...,...,...,...,...,...,...,...,...
11960,9f926ab63615690246b5937770b32e6751a17dc1cd318a...,2023-11-12 00:17:22+00:00,en,indiatimes.com,indiatimes.com,2023-11-10,West Bengal: Revellers bursting banned cracker...,https://timesofindia.indiatimes.com/city/kolka...
11961,d843a3f11005485bc7bc6a211b9d63b2ac02b3551901a4...,2023-11-12 00:16:34+00:00,en,hindustantimes.com,hindustantimes.com,2023-11-10,Delhi govt defers odd-even and taxi ban plan,https://www.hindustantimes.com/cities/delhi-ne...
11962,ec64c46c4292c02040430d1a0e34dca94d1add8f9889d4...,2023-11-12 00:16:29+00:00,en,indiatimes.com,indiatimes.com,2023-11-10,Strengthening the debt resolution system for b...,https://timesofindia.indiatimes.com/blogs/kemb...
11963,b8076acc1e15c18389e39b2bdb8f15790b54462254cbcc...,2023-11-12 00:15:55+00:00,en,thehindu.com,thehindu.com,2022-06-26,Maharashtra political crisis: The battle for t...,https://frontline.thehindu.com/dispatches/maha...


In [16]:
df_judgements.to_csv('judgements.csv')

### Indian Military Operations

In [17]:
my_query = '''
(india OR indian OR indians) 

AND

(military OR army OR navy OR "air force")

AND

("operation peace" OR "operation polo" OR "golden temple" OR "operation vijay" OR "operation ablaze" OR "operation riddle"
OR "operation nepal" OR "operation steeplechase" OR "operation cactus lily" OR "operation blue star" OR "operation woodrose"
OR "operation meghdoot" OR "operation shivalik" OR "operation black thunder" OR "operation mand" OR "operation bluebird"
OR "operation pawan" OR "operation night dominance" OR "operation rakshak" OR "operation vadhi pahar" OR "operation election"
OR "operation sarp vinash" OR "operation black tornado" OR "operation all out" OR "operation calm down" OR "operation devi shakti" 
OR "operation ganga" OR "operation sindoor" OR "operation sindhu" OR "operation goodwill" OR "operation good samaritan"
OR "operation cactus" OR "operation meghdoot" OR "operation trident" OR "operation python" OR "operation restore hope" 
OR "operation talwar" OR "operation parakram" OR "operation enduring freedom" OR "operation sukoon" OR "operation searchlight"
OR "operation nistar" OR "operation madad" OR "operation samudra setu" OR "operation poomalai" OR "operation safed sagar" 
OR "operation maitri" OR "operation sankat mochan" OR "operation insaniyat" OR "operation bandar" OR "operation dost"
OR "operation kaveri" OR "operation ajay")

AND NOT (cricket OR bollywood OR "stock market")
'''

In [18]:
results = search_api.story_count(my_query, start, end, source_ids=source_IDs)

print(results)

{'relevant': 10965, 'total': 10420074}


In [19]:
df_military = get_stories(my_api_key, source_IDs, my_query, start, end, results['relevant'])

df_military

Fetching stories: 10973story [08:24, 21.77story/s]                        


,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,38e0dbc63b7ed36a8fa9006fcf95370a0b5fc60c7e637a...,2025-07-02 16:18:25.190091+00:00,en,thequint.com,thequint.com,2019-06-20,Army to celebrate birth centenary of war hero ...,https://www.thequint.com/news/hot-news/army-to...
1,fc0376d7ce7d375256ecef91c8197b265389e6391db91b...,2025-07-02 15:19:14.717986+00:00,en,indiatimes.com,indiatimes.com,2019-06-21,'Operation Bandar' was IAF's code name for Bal...,https://economictimes.indiatimes.com/news/defe...
2,e6bd927786458d433666cba7b9296cb635fc7f7c8590fd...,2025-07-02 15:06:22.570227+00:00,en,opindia.com,opindia.com,2019-06-21,Balakot airstrikes was codenamed by IAF as 'Op...,https://www.opindia.com/2019/06/balakot-airstr...
3,7ecdc4a3b2fda98235ed1e2b55e576943158eb370389a4...,2025-07-02 14:44:59.912370+00:00,en,thehindu.com,thehindu.com,2019-06-21,"Operation Bandar, code name ofBalakot strike",https://www.thehindu.com/news/national/operati...
4,5c36a7b5ae46fd3376bf3a768263f28bee8fb8c69fb6b7...,2025-07-02 14:39:26.351866+00:00,en,thehindu.com,thehindu.com,2019-06-21,"Operation Bandar, code name of Balakot strike",https://www.thehindu.com/news/national/operati...
...,...,...,...,...,...,...,...,...
10968,b72736dd83a8ef319fe34c5141e9612f3a4a0fd917e413...,2023-11-18 01:28:37+00:00,en,thehindu.com,thehindu.com,2023-11-16,Indian Navy completes second anti-piracy patro...,https://www.thehindu.com/news/national/indian-...
10969,cb5468b05d230be1fe647715991533759acc93088eeba6...,2023-11-17 00:43:06+00:00,en,thehindu.com,thehindu.com,2023-11-15,"Evacuated from war-torn Ukraine, over 1000 Ind...",https://www.thehindu.com/news/international/ev...
10970,bb1c1487420ea21f277af5dd0083c0f9a54f998cc0d9ee...,2023-11-16 02:45:29+00:00,en,firstpost.com,firstpost.com,2023-11-14,Cash-strapped Pakistan sold weapons worth $364...,https://www.firstpost.com/world/cash-strapped-...
10971,6b632cfcd9ae606db6977f2fdf7af4a8da2fefcbbc41fd...,2023-11-13 03:45:00+00:00,en,rediff.com,rediff.com,2023-11-10,Pippa Review,https://www.rediff.com/movies/review/pippa-rev...


In [20]:
df_military.to_csv('military.csv')

### Foreign Summits and Policy

In [21]:
my_query = '''
(india OR indian OR indians) 

AND

("foreign policy" OR "foreign relations" OR "international relations" OR "india-* relations" OR "india * relations"
OR "indo-* relations" OR "indo * relations" OR "ministry of external affairs" OR "minister of external affairs")

AND

(ASEAN OR BRICS OR commonwealth OR EAS OR "east asia summit" OR G-4 OR G4 OR G-15 OR G15 OR G-20 OR G20 OR G-24 
OR G24 OR G-77 OR G77 OR IBRD OR "World Bank" OR ICAO OR "international chamber of commerce" OR "red cross" OR "red crescent"
OR IFC OR IMF OR "international monetary fund" OR Interpol OR olympic OR olympics OR NAM OR "non-aligned" OR "non-aligned movement" 
OR "non aligned movement" OR "non aligned" OR SAARC OR "south asian association for regional cooperation" OR UN 
OR "united nations" OR WTO OR "world trade organization")

AND NOT (cricket OR bollywood)

'''

In [22]:
results = search_api.story_count(my_query, start, end, source_ids=source_IDs)

print(results)

{'relevant': 17702, 'total': 10420074}


In [23]:
df_foreign = get_stories(my_api_key, source_IDs, my_query, start, end, results['relevant'])

df_foreign

Fetching stories: 17737story [13:01, 22.68story/s]                        


,id,indexed_date,language,media_name,media_url,publish_date,title,url
0,fe7ca470968b6ff0dd2ec063422e4af0cd5119059e3ec6...,2025-07-02 16:22:26.514033+00:00,en,thehindu.com,thehindu.com,2019-06-20,U.S. Secretary Mike Pompeo’s India visit to pa...,https://www.thehindu.com/news/national/us-secr...
1,ac97d55d26662b591df8010da80c9fabe40389c65cd418...,2025-07-02 16:14:11.272641+00:00,en,india.com,india.com,2019-06-21,"Ahead of India visit, US Secretary of State Mi...",https://zeenews.india.com/india/ahead-of-india...
2,da6a7f5887db4863c18a844dce05bb84017755dd9d8ab9...,2025-07-02 16:07:24.794179+00:00,en,india.com,india.com,2019-06-21,US Secretary of State Michael Pompeo to Visit ...,https://www.india.com/news/india/us-secretary-...
3,fb6ce2d94652d4c11f796568ce35eeeb12ff4ae52fbab9...,2025-07-02 16:00:26.996051+00:00,en,cnn.com,cnn.com,2018-09-24,Maldives election: Solih hails new dawn after ...,https://edition.cnn.com/2018/09/24/asia/maldiv...
4,0265dcd1bba026cbe24b5ce3bacc44e4caab7020b4ea30...,2025-07-02 16:00:21.943680+00:00,en,cnn.com,cnn.com,2018-09-24,Maldives election: Solih hails new dawn after ...,https://www.cnn.com/2018/09/24/asia/maldives-e...
...,...,...,...,...,...,...,...,...
17732,131bf728d8d6e03ef1d15d96c09208ffbbc80c620d6130...,2023-11-12 00:12:41+00:00,en,indiatimes.com,indiatimes.com,2023-11-10,PM Modi's state visit to US opened new chapter...,https://economictimes.indiatimes.com/news/indi...
17733,c3890e840ed2d1396e21b244749a78c2bb1b65b62f8f90...,2023-11-12 00:12:40+00:00,en,indiatimes.com,indiatimes.com,2023-11-10,S Jaishankar: PM Modi's state visit to US open...,https://timesofindia.indiatimes.com/india/pm-m...
17734,9bdbfdec2fd0876284f9ade4f0a33253f655cfd44fb776...,2023-11-12 00:11:56+00:00,en,india.com,india.com,2023-11-10,India-US Defence Cooperation Rock Solid Suppor...,https://zeenews.india.com/india/india-us-defen...
17735,7d6042babd4eec8efc1831af5c72e33856a3734d049a8b...,2023-11-12 00:11:29+00:00,en,india.com,india.com,2023-11-10,Strongest Ever...: Antony Blinken On India-US ...,https://zeenews.india.com/india/strongest-ever...


In [24]:
df_foreign.to_csv('foreign.csv')